# Tekne Dedektörü v4s_frozen_v2 — Google Colab GPU Eğitimi (genişletilmiş dataset)

Bu notebook, `boat_v4s_frozen` deneyini (YOLOv8s, backbone donuk `freeze=10`, imgsz=640) aynı tarifle ama **hard-negative/hard-positive mining ile genişletilmiş datasetle** (train: 2802 -> 3944 görüntü) baştan (fresh, resume değil) eğitir.

**Önce yapman gerekenler:**
1. Üstteki menüden **Çalışma zamanı (Runtime) > Çalışma zamanı türünü değiştir > T4 GPU** (veya A100/L4) seç.
2. `boat_v4s_frozen_v2_bundle.zip` dosyasını (Mac'indeki `depth-anything` klasöründe, ~384MB) Google Drive'ına yükle.
3. Aşağıdaki hücreleri sırayla çalıştır.

In [ ]:
# 1) GPU kontrolü
!nvidia-smi

In [ ]:
# 2) Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Zip dosyasını Drive'dan al ve aç
# Zip'i Drive'da farklı bir yere yüklediysen ZIP_PATH'i güncelle.
ZIP_PATH = '/content/drive/MyDrive/boat_v4s_frozen_v2_bundle.zip'

!rm -rf /content/work
!mkdir -p /content/work
!unzip -q "$ZIP_PATH" -d /content/work
!ls /content/work

In [ ]:
# 4) ultralytics kur
!pip install -q ultralytics

In [ ]:
# 5) dataset.yaml içindeki path'i Colab'daki yeni konuma göre düzelt
# (Mac'te: /Users/armin/Desktop/depth-anything/yolo_dataset_v4 idi)
import pathlib

yaml_path = pathlib.Path('/content/work/yolo_dataset_v4/dataset.yaml')
content = yaml_path.read_text()
print('--- eski ---')
print(content)

new_content = content.replace(
    '/Users/armin/Desktop/depth-anything/yolo_dataset_v4',
    '/content/work/yolo_dataset_v4'
)
yaml_path.write_text(new_content)
print('--- yeni ---')
print(yaml_path.read_text())

In [ ]:
# 6) backbone'un gercekten ilk 10 katmanda bittigini teyit et (Mac'te de dogrulanmisti)
from ultralytics import YOLO

_check = YOLO('/content/work/yolov8s.pt')
for i, layer in enumerate(_check.model.model):
    print(i, layer.__class__.__name__)

In [ ]:
# 7) Eğitim — boat_v4s_frozen ile AYNI tarif (freeze=10, imgsz=640, patience=25),
# tek fark: genisletilmis dataset. Fresh baslatiyoruz (resume degil) ki eski/yeni
# dataset karsilastirmasi ayni recipe uzerinden temiz olsun.
from ultralytics import YOLO

model = YOLO('/content/work/yolov8s.pt')
results = model.train(
    data='/content/work/yolo_dataset_v4/dataset.yaml',
    epochs=80,
    imgsz=640,
    device=0,
    batch=16,
    patience=25,
    freeze=10,
    project='/content/work/runs_boat_yolo',
    name='boat_v4s_frozen_v2',
    verbose=True,
)

In [ ]:
# 8) Eğitim koptuysa devam ettirmek için (7. hücre yerine bunu çalıştır):
# from ultralytics import YOLO
# model = YOLO('/content/work/runs_boat_yolo/boat_v4s_frozen_v2/weights/last.pt')
# results = model.train(resume=True)

In [ ]:
# 9) Bitince: sonuçları Drive'a kopyala
!mkdir -p /content/drive/MyDrive/boat_v4s_frozen_v2_results
!cp -r /content/work/runs_boat_yolo/boat_v4s_frozen_v2 /content/drive/MyDrive/boat_v4s_frozen_v2_results/
print('Kopyalandı: Google Drive > boat_v4s_frozen_v2_results > boat_v4s_frozen_v2')

## Eğitim bitince Mac'ine geri alma

1. Drive'daki `boat_v4s_frozen_v2_results/boat_v4s_frozen_v2` klasörünü indir (`weights/best.pt` şart, geri kalanı - grafikler/results.csv - isteğe bağlı ama karşılaştırma için faydalı).
2. Bana zip'i ilet, ben `runs/detect/runs_boat_yolo/boat_v4s_frozen_v2/` altına yerleştirip mevcut `boat_v4s_frozen` (freeze=10, eski dataset) ile karşılaştırırım.

**Not — Colab'ın ücretsiz kotası:** Oturumlar genelde ~12 saatte bir kesilir, uzun süre etkileşimsiz kalırsan daha erken de kopabilir. imgsz=640 + freeze=10, önceki denemede GPU'da 74 epoch ~50 dakika sürmüştü (epoch başına ~40sn) - 80 epoch tam olarak muhtemelen ~1 saat civarı.